In [45]:
import pandas as pd 
import os
import numpy as np
import datetime

In [46]:
# Make a list of all the csv file paths in the directories
data_loc = 'data/2014-2024_data/'
excel_file_names = [data_loc+file for file in os.listdir(data_loc) if '.xlsx' in file]


In [47]:
# creates a dictionary of DataFrames with key given by each sheet
list_of_dataframes = [pd.read_csv("data/TTC_Streetcar_Delay_Data_since_2025.csv")]
for file in excel_file_names:
    temp = pd.read_excel(file, sheet_name=None)
    list_of_dataframes += list(temp.values())


In [48]:
data_2014_2025 = pd.concat(list_of_dataframes, ignore_index=True)

In [49]:
#Figure out which columns have been renamed over the years and need to be merged 
#data_2014_2025.info()

It seems like at some point the ttc changed terminology, where :
- 'Min Gap' == 'Gap'
- 'Min Delay' == 'Delay'
- 'Bound' == 'Direction'
- 'Date' == 'Report Date' 
- 'Line' == 'Route'
- 'Incident' == 'Code'
- 'Location' == 'Station'


In order to verify these claims are true, let's make sure that there is 
never an instance where both columns are *not* NaN. 

In [50]:
possible_mergers = [('Min Gap', 'Gap'), ('Min Delay', 'Delay'), ('Bound', 'Direction'), ('Date', 'Report Date'), ('Line', 'Route'), ('Incident', 'Code'), ('Station', 'Location')]
for test1,test2 in possible_mergers:
    print(f'Number of colliding parameters when comparing {test1} with {test2} :')
    print(data_2014_2025[~data_2014_2025[test1].isna()  & ~data_2014_2025[test2].isna()].shape[0])

Number of colliding parameters when comparing Min Gap with Gap :
0
Number of colliding parameters when comparing Min Delay with Delay :
0
Number of colliding parameters when comparing Bound with Direction :
0
Number of colliding parameters when comparing Date with Report Date :
0
Number of colliding parameters when comparing Line with Route :
0
Number of colliding parameters when comparing Incident with Code :
0
Number of colliding parameters when comparing Station with Location :
0


It looks like we can merge all but 'Incident' with 'Incident ID'. We will 
try to conform to the terminology in the 2025 data. 

In [51]:
# We keep left tuple column and drop the right tuple column after merging
mergers = [('Min Gap', 'Gap'), ('Min Delay', 'Delay'), ('Bound', 'Direction'), ('Date', 'Report Date'), ('Line', 'Route'), ('Incident', 'Code'), ('Station', 'Location')]
for keep, destroy in mergers:
    # Because we have 'inplace=True', rerunning the cell will cause an error.
    # Therefore, we add a 'try' conditional. 
    try:
        data_2014_2025[keep] = data_2014_2025[keep].combine_first(data_2014_2025[destroy])
        data_2014_2025.drop(columns=[destroy], inplace=True)
    except:
        print('Cell did not run. Did you already run this cell?')

It looks like the _id column is a unique identifier for each instance and this only occurs in the 2025 data. We will verify that these values are all unique when they occur first. 

In [52]:
col_id = data_2014_2025['_id']

duplicates = data_2014_2025[col_id.notna() & col_id.duplicated()]
duplicates

,_id,Date,Line,Time,Day,Station,Min Delay,Min Gap,Bound,Vehicle,Incident,Incident ID


There are no duplicates in this column so we can drop it. We will also drop the column Incident ID, which only occurs in April 2019 which assigns a numerical value to each Incident.  

In [53]:
data_2014_2025 = data_2014_2025.drop(columns=['_id', 'Incident ID'])

Finally, we sort the dataframe by the Date and then the time, in lexicographic order. 

In [ ]:
# Strangely, some of the Time data is stamped with date. 
# In particular, these times are *always* listed as midnight, 
# So I will elect to convert these to NAs.
data_2014_2025[data_2014_2025.Time.apply(lambda x : isinstance(x, datetime.datetime))]

## I checked the first few of these in the raw data, and it looks like each entry like this is a duplicate of another entry with all values the same except for time. 

,Date,Line,Time,Day,Station,Min Delay,Min Gap,Bound,Vehicle,Incident
17107,2016-03-10 00:00:00,501.0,2016-03-10 00:00:00,Thursday,Queenway and Park lawn,36.0,44.0,B/W,4181.0,Diversion
31668,2020-05-21 00:00:00,509.0,2020-05-21 00:00:00,Thursday,Ferry Docks streetcar stop,1.0,2.0,W/B,4459.0,Investigation
31704,2020-05-23 00:00:00,505.0,2020-05-23 00:00:00,Saturday,Dundas West Station,15.0,20.0,E/B,4444.0,Mechanical
56539,2017-05-27 00:00:00,505.0,2017-05-27 00:00:00,Saturday,Broadview STN,12.0,17.0,W/B,4190.0,General Delay
145384,2019-08-05 00:00:00,510.0,2019-08-04 00:00:00,Monday,spadina station,6.0,13.0,S/B,4421.0,Mechanical
147254,2019-10-14 00:00:00,509.0,2019-10-14 00:00:00,Monday,Fleet Loop,8.0,22.0,E/B,4416.0,General Delay


In [55]:
def get_time(x):
    if type(x) == datetime.datetime:
        return pd.NA
    else:
        return x
    
data_2014_2025.Time = data_2014_2025.Time.apply(get_time)

In [ ]:
# Similarly, some of the Date data is stamped with a time. 
# In particular, these times are *always* listed as midnight, rather than the time in the time column.
# I will drop this component of the date time object in the Date column so that it only retains the date portion. 

data_2014_2025["Date"] = data_2014_2025["Date"].dt.date

Now we will reorder the columns to group time properties and location properties, and then sort by time/date.

In [64]:
data_2014_2025 = data_2014_2025[
    ['Date', 'Time', 'Day', 'Line', 'Station', 'Bound', 'Vehicle', 'Incident', 'Min Delay', 'Min Gap']
]
data_2014_2025.sort_values(by=['Date', 'Time'], inplace=True)

In [65]:
data_2014_2025.to_csv('data/2014-2025data.csv', index=False)